# Анализ продаж интернет-магазина

Интерактивный анализ выручки, стран, товаров и сезонности. Данные автоматически загружаются из UCI Machine Learning Repository.

In [ ]:
import io
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_URL = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'

with urlopen(DATA_URL) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    xlsx_file = next(name for name in archive.namelist() if name.endswith('.xlsx'))
    with archive.open(xlsx_file) as source:
        df = pd.read_excel(source)

df.head()

## Подготовка данных

Исключаем отменённые заказы, некорректные позиции и строки без клиента или названия товара.

In [ ]:
df.columns = df.columns.str.strip()
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df = df.dropna(subset=['CustomerID', 'Description']).copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)

print(f'Строк после очистки: {len(df):,}')
print(f'Выручка: £{df["Revenue"].sum():,.2f}')
print(f'Уникальных клиентов: {df["CustomerID"].nunique():,}')

## Ключевые результаты

In [ ]:
top_countries = (
    df.groupby('Country', as_index=False)['Revenue'].sum()
    .sort_values('Revenue', ascending=False).head(10)
)
top_products = (
    df.groupby('Description', as_index=False)['Revenue'].sum()
    .sort_values('Revenue', ascending=False).head(10)
)

display(top_countries)
display(top_products)

In [ ]:
sns.set_theme(style='whitegrid')
monthly_revenue = df.groupby('Month', as_index=False)['Revenue'].sum()
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_revenue, x='Month', y='Revenue', marker='o')
plt.title('Динамика выручки по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Выручка, £')
plt.xticks(rotation=45)
plt.show()

## Рекомендации для бизнеса

- Планировать увеличенные запасы ключевых товаров к осени и предновогоднему периоду.
- Активнее продвигать товары-лидеры, уже подтверждающие высокий спрос.
- Развивать продажи в Нидерландах, Ирландии, Германии и Франции.
- Снижать зависимость от рынка Великобритании за счёт международного роста.